<h1>Chapter 2 – Tokens and Token Embeddings</h1>
<p><i>From Words to Vectors: Teaching Machines the Language of Meaning.</i></p>

<p>
  <a href="https://github.com/bhattacharjeeajay12/hands-on-ai-ml" target="_blank">
    <img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github" alt="GitHub Repository">
  </a>

  <a href="https://colab.research.google.com/github/bhattacharjeeajay12/hands-on-ai-ml/blob/main/chapter01/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
  </a>
</p>

<hr>

<p>
This notebook is <strong>Chapter 2</strong> of the series
<strong>"AI-ML Learning by Doing"</strong> by
<a href="https://emberground.org/" target="_blank">EmberGround</a>.
</p>

<p>
<b>Authors:</b>
<a href="https://www.linkedin.com/in/ajay-bhattacharjee-60106651/" target="_blank">Ajay Bhattacharjee</a>
and
<a href="https://www.linkedin.com/in/dav-id-7b6160222/" target="_blank">David</a>.
</p>

<hr>

## 📑 Table of Contents

---
<!---->
### Section 1: Introduction to Tokenization
* [1.1 What is Tokenization](#11-why-tokenization-is-needed)
* [1.2 Vocabulary](#12-the-tokenization-pipeline-text-to-ids)
* [1.3 Word-Level Tokenization and Its Limitations](#13-the-tokenization-pipeline-text-to-ids)
* [1.4 Sub-Word Tokenization and it's benefits](#14-the-tokenization-pipeline-text-to-ids)
* [1.5 Classical NLP tokenisers Vs Modern tokenisers for LLM](#14-the-tokenization-pipeline-text-to-ids)


### Section 2: Sub-Word Tokenization
* [2.1 Tokens, Vocabulary, and Subwords](#21-tokens-vocabulary-and-subwords)
* [2.2 Popular Tokenizers (BPE, WordPiece, Unigram)](#22-popular-tokenizers-bpe-wordpiece-unigram)
* [2.3 From Tokens to Vectors (Embeddings)](#23-from-tokens-to-vectors-embeddings)

---

# Section 1: Introduction to Tokenisation
## <font color="#4188ff">1.1 What is Tokenization ?</font>

- Computers cannot process human words directly.
- They only understand numbers.
- Tokenization converts text into smaller units called **tokens**.
- Each token is mapped to a unique **ID (number)**.
- The LLM processes these token IDs instead of raw text.

**In one sentence:**

> **Tokenization converts human-readable text into numeric tokens that an LLM can understand and process efficiently.**

![Tokenization](https://raw.githubusercontent.com/bhattacharjeeajay12/hands-on-ai-ml/main/content/images/tokeniser1.png)

## <font color="#4188ff">1.2 Vocabulary</font>
Every unique token produced by a tokenizer belongs to that model's <font color="red">**vocabulary**</font>. Different models use different vocabularies and tokenizer algorithms.

| Model Name | Developer / Organization | Vocabulary Size | Tokenizer Type |
| :--- | :--- | :--- | :--- |
| **LLaMA 2** | Meta | **~32,000** tokens | SentencePiece (BPE) |
| **GPT-3.5 / GPT-4** | OpenAI | **~100,000** tokens | tiktoken (`cl100k_base`) |
| **LLaMA 3 / 3.1** | Meta | **~128,000** tokens | tiktoken-based |
| **Claude 3 / 3.5** | Anthropic | **~100,000+** tokens | Custom BPE |
| **Gemma / Gemma 2 / 3** | Google | **~256,000** tokens | SentencePiece |

## <font color="#4188ff">1.3: Word-Level Tokenization and Its Limitations</font>

### <font color="#2F855A">What is Word-Level Tokenization?</font>

In **Word-Level Tokenization**, the text is split purely by whitespace or punctuation, treating every unique full word as an individual token.

---

### <font color="#2F855A">How Older NLP Tried to Fix It: Stemming & Lemmatization</font>

To prevent the vocabulary from exploding, classical NLP systems used rule-based preprocessing steps **before** tokenizing:

* **Stemming (Chopping):** Crudely cuts off word endings using hardcoded rules.
  * `"running"`, `"runs"`, `"runner"` $\rightarrow$ **`"run"`**
* **Lemmatization (Dictionary Lookup):** Uses grammar and dictionaries to reduce words to their base form (*lemma*).
  * `"better"` $\rightarrow$ **`"good"`** | `"was"` $\rightarrow$ **`"be"`**

### Why modern LLMs abandoned Stemming & Lemmatization:
1. **Loss of Nuance & Context:** Converting `"was"` to `"be"` strips away important tense and tone information needed for natural generation.
2. **Slow & High-Maintenance:** Require complex grammatical rules and huge dictionaries for every single language.
3. **Cannot Handle Misspellings or Code:** They fail on typos, brand names, or programming languages (e.g., Python code).

---

### <font color="#2F855A">Why Word-Level Tokenization Fails for Modern LLMs</font>

Despite classical NLP techniques like **stemming** and **lemmatization**, word-level tokenization is **not suitable for modern LLMs** for the following reasons:

### 1. Massive Vocabulary Size
A word-level tokenizer requires every unique word to be stored in its vocabulary. Across multiple languages, domain-specific terms, names, and newly coined words, the vocabulary quickly grows to millions of words, making the model larger and less efficient.

### 2. Out-of-Vocabulary (OOV) Words
If the model encounters a new word, typo, brand name, or recently coined term that is not in its vocabulary, it is treated as an **`<UNK>` (Unknown)** token, causing the model to lose important information.

**Example:** `ChatGPT`, `LangGraph`, `DeepSeek`, `tokenization-aware`

### 3. Poor Generalization to New Words
Word-level tokenization cannot infer the meaning of unseen words from familiar parts. For example, if the model knows **"token"** but has never seen **"tokenization"** or **"tokenizer"**, it treats them as completely different words.

Sub-word tokenization instead splits them into:

```text
tokenization → token + ization
tokenizer    → token + izer
```

allowing the model to generalize to new words.

### 4. Inefficient for Programming Languages
Modern LLMs process both **natural language and source code**. Code contains identifiers such as:

```python
calculate_embedding_similarity()
```

A word-level tokenizer would treat the entire identifier as one token, whereas a sub-word tokenizer can split it into meaningful pieces like:

```text
calculate + _ + embedding + _ + similarity
```

This enables the model to understand previously unseen variable names, function names, and class names.

### 5. Poor Support for Morphologically Rich Languages
Many languages (e.g., German, Turkish, and Finnish) create long compound words or have numerous word forms. A word-level vocabulary would need to store each variation separately, making it impractical.

---

> **Key Takeaway for Students:** Older systems used **stemming and lemmatization** as "band-aids" to shrink the vocabulary, but they destroyed word nuances. Modern LLMs replaced all of this with **sub-word tokenization**—which handles prefixes, suffixes, typos, and new words naturally without stripping away grammar or meaning!

## <font color="#4188ff">1.4: Exercise - Drawbacks of Word Level Tokenization</font>

In [16]:
import json
import os
import nltk

JSON_FILE = "nltk_vocab.json"

# Run this once to generate the JSON cache
if not os.path.exists(JSON_FILE):
    print("Downloading NLTK words for the first time...")
    nltk.download('words', quiet=True)
    from nltk.corpus import words

    # Save as JSON list
    vocab_list = words.words()
    with open(JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(vocab_list, f)
    print(f"Saved {len(vocab_list):,} words to {JSON_FILE}!")

# Fast Load: Loads in milliseconds from your local disk
with open(JSON_FILE, "r", encoding="utf-8") as f:
    nltk_vocab = set(json.load(f))

print(f"Loaded {len(nltk_vocab):,} words instantly!")

Saved 236,736 words to nltk_vocab.json!
Loaded 235,892 words instantly!


In [19]:
# =====================================================================
# STEP 1: Setup and Preprocessing (Classical NLP)
# =====================================================================
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK resources silently
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Define a small hypothetical "Vocabulary Dictionary" our model knows
# KNOWN_VOCABULARY = {"token", "run", "cat", "study", "think", "text", "<UNK>"}
KNOWN_VOCABULARY = nltk_vocab

# Sample words to test
test_words = ["running", "studying", "tokenizing", "untokenizable", "Mahindra"]

print("=== 1. CLASSICAL APPROACH: STEMMING & LEMMATIZATION ===")
print(f"Model's Known Vocabulary ({len(KNOWN_VOCABULARY)} items): {KNOWN_VOCABULARY}\n")

# Process each word and check for OOV
for word in test_words:
    stemmed = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word, pos="v") # pos='v' treats words as verbs

    # Check if the stemmed/lemmatized word exists in our dictionary
    status_stem = stemmed if stemmed in KNOWN_VOCABULARY else "<UNK> (OOV Error!)"
    status_lemma = lemmatized if lemmatized in KNOWN_VOCABULARY else "<UNK> (OOV Error!)"

    print(f"Original: '{word}'")
    print(f"  ├─ Stemmed:    '{stemmed}'    --> Lookup result: {status_stem}")
    print(f"  └─ Lemmatized: '{lemmatized}' --> Lookup result: {status_lemma}")
    print("-" * 55)

# =====================================================================
# STEP 2: Modern Solution (Sub-Word Tokenization with tiktoken / BPE)
# =====================================================================
print("\n=== 2. MODERN APPROACH: SUB-WORD TOKENIZATION (GPT Tokenizer) ===")

import tiktoken

# Load OpenAI's standard sub-word tokenizer
enc = tiktoken.get_encoding("cl100k_base")

for word in test_words:
    token_ids = enc.encode(word)
    subword_pieces = [enc.decode_single_token_bytes(t).decode('utf-8') for t in token_ids]

    print(f"Original: '{word}'")
    print(f"  ├─ Token IDs:     {token_ids}")
    print(f"  └─ Sub-word Pieces: {subword_pieces}")
    print("-" * 55)

=== 1. CLASSICAL APPROACH: STEMMING & LEMMATIZATION ===
Model's Known Vocabulary (235892 items): {'ambulacral', 'somnambulism', 'corrupting', 'Ova', 'Heterosporeae', 'waipiro', 'keto', 'clinostat', 'submountain', 'unimaginableness', 'superacknowledgment', 'respectless', 'expander', 'clothespin', 'hoarily', 'overslur', 'commensalist', 'uproariously', 'steamcar', 'kettle', 'overproudly', 'dultie', 'naseberry', 'geniture', 'coly', 'sculptural', 'tachometry', 'grindle', 'Teresina', 'schizopelmous', 'unfashionably', 'taffy', 'peasant', 'decagonal', 'Rangifer', 'metameric', 'prereckon', 'avidous', 'droopingness', 'unbigoted', 'cathedraled', 'Pheny', 'Knossian', 'thrift', 'obstetrication', 'flaunty', 'tableclothwise', 'vangeli', 'antidote', 'upridge', 'curwhibble', 'tumbleweed', 'snuff', 'punchinello', 'dichotomistic', 'unvizarded', 'sottish', 'journalish', 'tractorize', 'distinguishingly', 'illatively', 'shantyman', 'fluidize', 'labiomancy', 'pouringly', 'ultratrivial', 'agrimotor', 'Peterlo

Loading vocabulary...
Loaded 370,100 words successfully!
Is 'tokenization' in dictionary? False


In [15]:
len(english_vocab)

370100

## <font color="#4188ff">Section 1.5: Sub-Word Tokenization and it's Benefits</font>

Play with these example sentences in the <a href="https://tiktokenizer.vercel.app/?model=gpt2" target="_blank">tiktokenizer app</a>:

> **"The tokenizer is tokenizing tokenizable text unexpectedly."**

> **"Mahindra loves overgeneralizing complex stuffs."**

---

### <font color="#2F855A">1. How Sub-Word Tokenization Splits Text</font>

Notice how the tokenizer breaks down variations of common roots and rare words into standard building blocks:

| Original Word | Sub-Word Tokens | What's Happening Here? |
| :--- | :--- | :--- |
| **`tokenizer`** | `["token", "izer"]` | Reuses root **`token`** + common suffix **`izer`** |
| **`tokenizing`** | `["token", "izing"]` | Reuses root **`token`** + common suffix **`izing`** |
| **`tokenizable`** | `["token", "iz", "able"]` | Reuses root **`token`** + suffixes **`iz`** + **`able`** |
| **`Mahindra`** | `["Mah", "ind", "ra"]` | Rare proper noun split into smaller recognized sub-units |
| **`overgeneralizing`** | `["over", "general", "izing"]` | Split into prefix **`over`** + root **`general`** + suffix **`izing`** |

---

### <font color="#2F855A">2. Why This is Superior for LLMs</font>

Sub-word tokenization solves the major problems of word-level tokenization through four key advantages:

1. **Efficiency (Keeps Common Words Intact):**
   - High-frequency words like `"The"`, `"is"`, or `"text"` remain single whole tokens, saving memory and processing speed.

2. **Smart Meaning Reuse (Morphological Understanding):**
   - Because `"token"` is stored separately from `"izing"` or `"izer"`, the LLM understands that `tokenizing` and `tokenizable` share the exact same root concept.

3. **Zero Out-of-Vocabulary (OOV) Errors:**
   - Brand-new words, rare names (`"Mahindra"`), or typos (`"runnning"`) won't crash or trigger an `<UNK>` (Unknown) error. The model simply stitches them together using smaller pieces.

4. **Compact Vocabulary Size:**
   - The model can represent millions of English words using a tiny, fixed set of around **32,000 to 100,000 sub-word tokens**.

---

> **Analogy:** Think of sub-word tokenization like **Lego blocks**. Instead of forcing the computer to store a custom pre-built toy for every single word in existence, it keeps a set of small, reusable Lego bricks (prefixes, roots, suffixes) that can build *any word imaginable*.